In [1]:
import pandas as pd
import yfinance as yf
import os

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import plotly.graph_objs as go
import dash
from dash import dcc, html

Create Directory if needed

In [2]:
os.makedirs(os.path.dirname("./saveData"), exist_ok=True)

Download Twitter Data

In [3]:
sentiment_path = "./savedData/datasets/gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged/versions/1/mbsa.csv"

if os.path.exists(sentiment_path):
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data loaded")
else:
    os.environ["KAGGLEHUB_CACHE"] = "./savedData"
    import kagglehub

    # Download the Bitcoin tweet dataset
    path = kagglehub.dataset_download("gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged")
    print("Path to dataset files:", path)
    df_sentiment = pd.read_csv(sentiment_path)
    print("Sentiment data downloaded, saved and loaded")

Sentiment data loaded


Data Pipeline

In [4]:
# Delete Duplicates
df_sentiment = df_sentiment.drop_duplicates()

# Clean Dates
df_sentiment = df_sentiment[pd.to_datetime(df_sentiment['Date'], errors='coerce').notnull()]
df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'], errors='coerce')

# Clean and map Sentiment
sentiment_map = {'Positive': 1, 'Negative': -1}
df_sentiment = df_sentiment.dropna(subset=['Sentiment'])
df_sentiment = df_sentiment[df_sentiment['Sentiment'].isin(['Positive', 'Negative'])].copy()
df_sentiment['sentiment_score'] = df_sentiment['Sentiment'].map(sentiment_map)

# Clean Text
df_sentiment = df_sentiment.dropna(subset=['text'])

Optional: Remove emojis and url´s from tweets

In [5]:
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+", '', x))
# df_sentiment.loc[:, 'text'] = df_sentiment['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))

In [6]:
start_date = df_sentiment['Date'].min()
end_date = df_sentiment['Date'].max()
print("Minimales Datum: ", start_date, "\nMaximales Datum: ", end_date)

Minimales Datum:  2014-09-18 00:00:00 
Maximales Datum:  2019-11-23 00:00:00


Download Bitcoin Data

In [7]:
bitcoin_path = "./savedData/bitcoinPrice.csv"

if(os.path.exists(bitcoin_path)):
    df_bitcoin = pd.read_csv(bitcoin_path, header=0)
    df_bitcoin = df_bitcoin.dropna(subset=["Date"])
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    print("Bitcoin data loaded")
else:
    ticker = "BTC-USD"
    df_bitcoin = yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    df_bitcoin.reset_index(inplace=True)
    df_bitcoin = df_bitcoin[['Date', 'Close']]
    df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])
    df_bitcoin.columns = ['Date', 'Close']
    df_bitcoin.to_csv(bitcoin_path, index=False)
    print("Bitcoin data downloaded, loaded and saved")

Bitcoin data loaded


In [8]:
# Calculate sentiment per day
df_daily_sentiment = df_sentiment.groupby('Date')['sentiment_score'].mean().reset_index()
df_daily_sentiment.rename(columns={'sentiment_score': 'avg_sentiment'}, inplace=True)
df_daily_sentiment['Date'] = pd.to_datetime(df_daily_sentiment['Date'])

Complete Sentiment DF with next day price and avgSentiment

In [9]:
df_sentiment = df_sentiment.merge(df_daily_sentiment, on='Date', how='left')
df_bitcoin['next_day_price'] = df_bitcoin['Close'].shift(-1)
df_sentiment = df_sentiment.merge(df_bitcoin[['Date', 'next_day_price']], on='Date', how='left')

In [10]:
df_sentiment_path = "./savedData/df_sentiment.csv"

if os.path.exists(df_sentiment_path):
    df_sentiment = pd.read_csv(df_sentiment_path)
    print("Already Saved")
else:
    df_sentiment.to_csv(df_sentiment_path, index=False)
    print("Saved")

Already Saved


In [11]:
df_sentiment.head()

,Date,text,Sentiment,sentiment_score,avg_sentiment,next_day_price
0,2019-05-27,È appena uscito un nuovo video! LES CRYPTOMONN...,Positive,1,1.0,8719.961914
1,2019-05-27,Cardano: Digitize Currencies; EOS https://t.co...,Positive,1,1.0,8719.961914
2,2019-05-27,Another Test tweet that wasn't caught in the s...,Positive,1,1.0,8719.961914
3,2019-05-27,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,Positive,1,1.0,8719.961914
4,2019-05-27,Spiv (Nosar Baz): BITCOIN Is An Asset &amp; NO...,Positive,1,1.0,8719.961914


In [12]:
# Nach deinem letzten Merge-Schritt hinzufügen:

# Sentiment-Features pro Tag
df_daily_features = df_sentiment.groupby('Date').agg({
    'sentiment_score': ['mean', 'std', 'min', 'max'],
    'text': 'count'  # Tweet-Volumen
}).reset_index()

df_daily_features.columns = ['Date', 'avg_sentiment', 'sentiment_volatility', 
                              'min_sentiment', 'max_sentiment', 'tweet_volume']

# Positive/Negative Tweet Ratio
sentiment_counts = df_sentiment.groupby('Date')['sentiment_score'].apply(
    lambda x: (x > 0).sum() / len(x)
).reset_index()
sentiment_counts.columns = ['Date', 'positive_ratio']

df_daily_features = df_daily_features.merge(sentiment_counts, on='Date')

# Zuerst: Datentyp-Konvertierung für df_bitcoin
df_bitcoin['Close'] = pd.to_numeric(df_bitcoin['Close'], errors='coerce')
df_bitcoin = df_bitcoin.dropna(subset=['Close'])  # Entferne ungültige Werte

# Dann: Feature-Engineering
df_bitcoin['price_change_pct'] = df_bitcoin['Close'].pct_change()
df_bitcoin['volatility_7d'] = df_bitcoin['Close'].rolling(7).std()
df_bitcoin['ma_7d'] = df_bitcoin['Close'].rolling(7).mean()
df_bitcoin['ma_30d'] = df_bitcoin['Close'].rolling(30).mean()

# Zielvariable
df_bitcoin['next_day_price'] = df_bitcoin['Close'].shift(-1)
df_bitcoin['price_direction'] = (df_bitcoin['next_day_price'] > df_bitcoin['Close']).astype(int)

# Zielvariable: Preis-Richtung (für Klassifikation)
df_bitcoin['price_direction'] = (df_bitcoin['next_day_price'] > df_bitcoin['Close']).astype(int)


In [13]:
df_daily_features.head()

,Date,avg_sentiment,sentiment_volatility,min_sentiment,max_sentiment,tweet_volume,positive_ratio
0,2014-09-18,-1.0,0.0,-1,-1,593,0.0
1,2014-09-19,-1.0,0.0,-1,-1,635,0.0
2,2014-09-20,1.0,0.0,1,1,641,1.0
3,2014-09-21,-1.0,0.0,-1,-1,577,0.0
4,2014-09-22,1.0,0.0,1,1,671,1.0


In [16]:
# Zuerst: Konvertiere beide Date-Spalten zu datetime
df_daily_features['Date'] = pd.to_datetime(df_daily_features['Date'])
df_bitcoin['Date'] = pd.to_datetime(df_bitcoin['Date'])

# Dann: Merge
df_model = df_daily_features.merge(df_bitcoin[['Date', 'Close', 'price_direction']], on='Date', how='left')

# Entferne NaN-Werte
df_model = df_model.dropna()

# Features und Zielvariable
feature_cols = ['avg_sentiment', 'sentiment_volatility', 'min_sentiment', 
                'max_sentiment', 'tweet_volume', 'positive_ratio']
X = df_model[feature_cols]
y = df_model['price_direction']

# Train-Test Split (80/20, chronologisch)
split_idx = int(len(df_model) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# Modell trainieren
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Vorhersagen
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Metriken
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance)


Accuracy: 0.5224

Classification Report:
              precision    recall  f1-score   support

         0.0       0.51      0.43      0.47       183
         1.0       0.53      0.61      0.57       196

    accuracy                           0.52       379
   macro avg       0.52      0.52      0.52       379
weighted avg       0.52      0.52      0.52       379


Feature Importance:
                feature  importance
4          tweet_volume    0.996722
0         avg_sentiment    0.001002
2         min_sentiment    0.000873
3         max_sentiment    0.000825
5        positive_ratio    0.000578
1  sentiment_volatility    0.000000


In [17]:
X.shape

(1891, 6)

In [18]:
# Dash Visualisierung
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1(f'Bitcoin Price Direction Prediction - Accuracy: {accuracy:.2%}'),
    
    # Confusion Matrix
    dcc.Graph(
        id='confusion-matrix',
        figure={
            'data': [
                go.Heatmap(
                    z=confusion_matrix(y_test, y_pred),
                    x=['Predicted Down', 'Predicted Up'],
                    y=['Actual Down', 'Actual Up'],
                    colorscale='Blues',
                    text=confusion_matrix(y_test, y_pred),
                    texttemplate='%{text}',
                    textfont={"size": 20}
                )
            ],
            'layout': go.Layout(
                title='Confusion Matrix',
                xaxis={'title': 'Predicted'},
                yaxis={'title': 'Actual'}
            )
        }
    ),
    
    # Feature Importance
    dcc.Graph(
        id='feature-importance',
        figure={
            'data': [
                go.Bar(
                    x=feature_importance['importance'],
                    y=feature_importance['feature'],
                    orientation='h',
                    marker=dict(color='steelblue')
                )
            ],
            'layout': go.Layout(
                title='Feature Importance',
                xaxis={'title': 'Importance'},
                yaxis={'title': 'Feature'},
                height=400
            )
        }
    ),
    
    # Prediction Probability Over Time
    dcc.Graph(
        id='prediction-probability',
        figure={
            'data': [
                go.Scatter(
                    x=df_model['Date'][split_idx:],
                    y=y_pred_proba,
                    mode='lines',
                    name='Predicted Probability (Up)',
                    line=dict(color='blue')
                ),
                go.Scatter(
                    x=df_model['Date'][split_idx:],
                    y=y_test.values,
                    mode='markers',
                    name='Actual Direction',
                    marker=dict(color='red', size=4)
                )
            ],
            'layout': go.Layout(
                title='Prediction Probability Over Time',
                xaxis={'title': 'Date'},
                yaxis={'title': 'Probability / Actual (0=Down, 1=Up)'},
                hovermode='closest'
            )
        }
    ),
    
    # Rolling Accuracy
    dcc.Graph(
        id='rolling-accuracy',
        figure={
            'data': [
                go.Scatter(
                    x=df_model['Date'][split_idx:],
                    y=pd.Series(y_test.values == y_pred).rolling(30, min_periods=1).mean(),
                    mode='lines',
                    name='30-Day Rolling Accuracy',
                    line=dict(color='green')
                )
            ],
            'layout': go.Layout(
                title='Rolling Accuracy (30-Day Window)',
                xaxis={'title': 'Date'},
                yaxis={'title': 'Accuracy'},
                hovermode='closest'
            )
        }
    )
])

if __name__ == '__main__':
    app.run_server(debug=True, port=8050)

